# Cell Type Neighborhoods v1

In [ ]:
import scanpy
import squidpy
import sklearn
import statsmodels

print(f"scanpy: {scanpy.__version__}")
print(f"squidpy: {squidpy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"statsmodels: {statsmodels.__version__}")

In [ ]:
# Autoreload extension for IPython/Jupyter
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('load_ext', 'autoreload')
        ipython.run_line_magic('autoreload', '2')
except (ImportError, NameError):
    pass

import pandas as pd
import numpy as np
import anndata as ad
import s3fs

import scanpy as sc
import matplotlib.pyplot as plt
import openslide
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter, AllDataCenters

sc.set_figure_params(dpi=150, color_map="viridis_r")
sc.settings.verbosity = 0

from cloudpathlib import S3Path
from scipy.sparse import issparse
from joblib import Parallel, delayed
from tqdm import tqdm

from gbmcollaboration.definitions import (
    DATA_PATH_CHUV_FILES,
    DATA_PATH_UKER_FILES,
    DATA_PATH_CR_FILES,
)
from gbmcollaboration import AllDataCenters

### Load Clinical Data

In [ ]:
import pandas as pd
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter

# Load processed clinical tables for all centers and combine into one cohort

def load_processed_clinical(dc, center_name):
    d = dc.load_tabular(exclude_idh1_mut=False)
    clin = d["clinical"]
    key = next((k for k in clin if "processed" in str(k).lower() and "clinical" in str(k).lower()), None)
    df = clin[key].copy() if key else list(clin.values())[0].copy()
    df["center"] = center_name
    return df

ch = load_processed_clinical(CHUVDataCenter, "CHUV")
uk = load_processed_clinical(UKERDataCenter, "UKER")
cr = load_processed_clinical(CRDataCenter, "Charite")

clin_all = pd.concat([ch, uk, cr], ignore_index=True, sort=False)
clin_out = clin_all.copy()

print("Total rows:", len(clin_out))
print("Totals by center:\n", clin_out.groupby("center").size())

### Load Visium Data

In [ ]:
adata = AllDataCenters.load_spatial_adata(
    exclude_low_quality_visium=True,
    exclude_idh1_mut=True,
    list_sample_id=None,
    normalization="log_norm_count",
    radius=1,
    deconvolution_slot_name="deconv_cohort_level_4",
)

print(adata)

### C2L Total Abundance Loader

In [ ]:
# Load ALL C2L abundance data (CHUV, UKER, Charite) - COMBINED
import gc
from joblib import Parallel, delayed
from joblib.externals.loky import get_reusable_executor

def load_sample(center_name, sample_id, h5ad_file):
    """Load and process a single sample using Cell2LocationLoader approach."""
    try:
        if isinstance(h5ad_file, S3Path) or str(h5ad_file).startswith('s3://'):
            s3 = s3fs.S3FileSystem(anon=False)
            h5ad_path_str = str(h5ad_file).replace('s3://', '')
            with s3.open(h5ad_path_str, 'rb') as f:
                adata = ad.read_h5ad(f)
        else:
            adata = ad.read_h5ad(h5ad_file)
        
        if 'means_cell_abundance_w_sf' not in adata.obsm:
            return None
        
        celltype_names = None
        if 'mod' in adata.uns and 'factor_names' in adata.uns['mod']:
            celltype_names = adata.uns['mod']['factor_names']
        elif 'factor_names' in adata.uns:
            celltype_names = adata.uns['factor_names']
        elif 'cell_type_names' in adata.uns:
            celltype_names = adata.uns['cell_type_names']
        
        if celltype_names is None:
            n_celltypes = adata.obsm['means_cell_abundance_w_sf'].shape[1]
            celltype_names = [f'celltype_{i}' for i in range(n_celltypes)]
        
        celltype_names = [f'meanscell_abundance_w_sf_{name}' for name in celltype_names]
        
        cell_abundance = adata.obsm['means_cell_abundance_w_sf']
        if issparse(cell_abundance):
            cell_abundance = cell_abundance.toarray()
        
        df = pd.DataFrame(cell_abundance, index=adata.obs_names, columns=celltype_names)
        df = df.fillna(0)
        df.index = pd.MultiIndex.from_tuples(
            [(sample_id, idx) for idx in df.index],
            names=['sample', 'barcode']
        )
        df['total_cells'] = df.sum(axis=1)
        
        return df
    except Exception as e:
        return None


def load_center(center_name, cell2loc_path):
    """Load one center with original fast parallel, then kill workers."""
    print(f"\nProcessing {center_name}...")
    base_path = S3Path(str(cell2loc_path))
    sample_dirs = [d.name for d in base_path.iterdir() if d.is_dir()]
    print(f"  Found {len(sample_dirs)} sample dirs for {center_name}")
    
    samples_to_process = [
        (center_name, sample_id, base_path / sample_id / "cohort_level_4.h5ad")
        for sample_id in sample_dirs
    ]
    
    # Original fast parallel call
    center_results = Parallel(n_jobs=-1)(
        delayed(load_sample)(center_name, sample_id, h5ad_file)
        for center_name, sample_id, h5ad_file in tqdm(samples_to_process, desc=f"  {center_name}")
    )
    
    center_results = [df for df in center_results if df is not None]
    
    for df in center_results:
        new_index = pd.MultiIndex.from_tuples(
            [(center_name, *idx) for idx in df.index],
            names=["center", "sample", "barcode"],
        )
        df.index = new_index
        df["center"] = center_name
    
    df_center = pd.concat(center_results, axis=0)
    
    print(f"  Loaded {df_center.index.get_level_values('sample').nunique()} samples from {center_name}")
    print(f"  Total spots: {df_center.shape[0]}")
    
    # FORCE KILL parallel workers to prevent hang on next call
    get_reusable_executor().shutdown(wait=True)
    gc.collect()
    
    return df_center


# Load each center (workers killed between each)
df_chuv = load_center("CHUV", DATA_PATH_CHUV_FILES["Cell2location"])
df_uker = load_center("UKER", DATA_PATH_UKER_FILES["Cell2location"])
df_cr = load_center("CR", DATA_PATH_CR_FILES["Cell2location"])


# Merge all centers
df_all = pd.concat([df_chuv, df_uker, df_cr], axis=0)

print("\n" + "=" * 60)
print("Combined df_all summary:")
print("  Total spots:", df_all.shape[0])
print("  Total samples:", df_all.index.get_level_values("sample").nunique())
print("  Index levels:", df_all.index.names)
print("  Example columns:", df_all.columns[:5].tolist())
print("=" * 60)

### Merge C2L and Visium Data

In [ ]:
# Drop the 'center' level from df_all's index for alignment
if isinstance(df_all.index, pd.MultiIndex) and "center" in df_all.index.names:
    df_all_no_center = df_all.copy()
    df_all_no_center.index = df_all_no_center.index.droplevel("center")
else:
    df_all_no_center = df_all

print("df_all_no_center index names:", df_all_no_center.index.names)
print("df_all_no_center shape:", df_all_no_center.shape)

# Build (sample, barcode) keys for each obs row, in obs order
barcodes = pd.Index(adata.obs_names).str.split("_").str[-1]
keys = pd.MultiIndex.from_arrays(
    [adata.obs["sample"].values, barcodes.values],
    names=["sample", "barcode"],
)

print("First 3 keys:", list(keys[:3]))
print("Number of keys:", len(keys))

# Align df_all_no_center to these keys
df_all_aligned = df_all_no_center.reindex(keys)
print("df_all_aligned shape:", df_all_aligned.shape)

# Drop columns from df_all_aligned that already exist in adata.obs
cols_to_add = [c for c in df_all_aligned.columns if c not in adata.obs.columns]
df_to_add = df_all_aligned[cols_to_add].copy()
df_to_add.index = adata.obs.index

# Join instead of concat to avoid fragmentation
adata.obs = adata.obs.join(df_to_add)

print(f"\nNumber of added columns: {len(cols_to_add)}")
print(f"Skipped (already exist): {len(df_all_aligned.columns) - len(cols_to_add)}")
print(f"adata.obs shape: {adata.obs.shape}")

### Load Custom Cell Annotation

In [ ]:
import pandas as pd

# user vars
KEY_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Branch_infercnv_visium_backup/analysis/SingleCell_Annotation_Key.csv"
LEVEL4_KEY = "celltype_level4_scanvi"
NEW_COLS = [
    "celltype_level4_curated",
    "celltype_level3_curated",
    "celltype_level3_figure",
    "celltype_level2_curated",
    "celltype_level2_figure",
    "celltype_level1_curated",
]

# 1) Load key
key_raw = pd.read_csv(KEY_PATH)

# 2) Basic column checks
required_cols = [LEVEL4_KEY] + NEW_COLS
missing_cols = [c for c in required_cols if c not in key_raw.columns]
if missing_cols:
    raise ValueError(f"Missing columns in key file: {missing_cols}")

# 3) Deduplicate on the level4 key and standardize to string
key_unique = key_raw.drop_duplicates(subset=[LEVEL4_KEY]).copy()
key_unique[LEVEL4_KEY] = key_unique[LEVEL4_KEY].astype(str)

# 4) Build a tidy DataFrame indexed by level4
celltype_key = key_unique.set_index(LEVEL4_KEY)[NEW_COLS]
celltype_key = celltype_key.apply(lambda s: s.astype(str))

# 5) Attach as the canonical celltype hierarchy in adata.uns
adata.uns["celltype_annotation_key"] = celltype_key

# 6) Quick coverage checks vs Visium abundance columns
lvl4_in_key = pd.Index(celltype_key.index)

obs_cols = pd.Index(adata.obs.columns)

# deconv_cohort_level_4 columns should use the raw level4 names
deconv_level4_cols = obs_cols.intersection(lvl4_in_key)

# C2L totals use "meanscell_abundance_w_sf_<level4>"
c2l_cols = [c for c in obs_cols if c.startswith("meanscell_abundance_w_sf_")]
c2l_lvl4 = pd.Index([c.replace("meanscell_abundance_w_sf_", "") for c in c2l_cols])
c2l_in_key = c2l_lvl4.intersection(lvl4_in_key)

missing_for_deconv = lvl4_in_key.difference(deconv_level4_cols)
missing_for_c2l = lvl4_in_key.difference(c2l_in_key)

print("Stored celltype_annotation_key in adata.uns['celltype_annotation_key']")
print(f"Total level4 types in key: {len(lvl4_in_key)}")
print(f"Matching deconv level4 columns in adata.obs: {len(deconv_level4_cols)}")
print(f"Matching C2L level4 types (from meanscell_abundance_w_sf_*): {len(c2l_in_key)}")

if len(missing_for_deconv):
    print(f"Level4 types in key not seen as deconv columns: {len(missing_for_deconv)}")
if len(missing_for_c2l):
    print(f"Level4 types in key not seen in C2L columns: {len(missing_for_c2l)}")

### Cluster Spots Based on Cell Type Abundance

In [ ]:
# %% [markdown]
# ### Cluster Spots Based on Cell Type Abundance

# %%
%matplotlib inline
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import gc

# User config
EXCLUDE_CELLTYPES = []
CHOSEN_K          = 20
CHOSEN_RES        = 0.6
celltype_rep_key  = "X_celltype"
umap_min_dist     = 0.5
umap_spread       = 1.2
random_state      = 0

# Consistent key formatting downstream (e.g. "celltype_leiden_k20_r0.6")
res_str           = f"{CHOSEN_RES:g}"
neighbors_key     = f"celltype_neighbors_k{CHOSEN_K}"
cluster_key       = f"celltype_leiden_k{CHOSEN_K}_r{res_str}"
umap_key          = "X_umap_celltype"

# Build cell type abundance matrix from C2L obs columns
c2l_prefix   = "meanscell_abundance_w_sf_"
celltype_key = adata.uns["celltype_annotation_key"]

c2l_cols    = [c for c in adata.obs.columns if c.startswith(c2l_prefix)]
lvl4_in_key = [
    c.replace(c2l_prefix, "") for c in c2l_cols
    if c.replace(c2l_prefix, "") in celltype_key.index
    and c.replace(c2l_prefix, "") not in EXCLUDE_CELLTYPES
]
c2l_cols_filt = [c2l_prefix + n for n in lvl4_in_key]

print(f"Cell types: {len(c2l_cols)} total, {len(lvl4_in_key)} used "
      f"({len(c2l_cols) - len(lvl4_in_key)} excluded or unmatched)")

X_ct = adata.obs[c2l_cols_filt].to_numpy(dtype=np.float32)

# Diagnostics: how prevalent are NaNs?
n_nan_cells = np.isnan(X_ct).sum()
n_nan_rows  = np.isnan(X_ct).any(axis=1).sum()
n_nan_cols  = np.isnan(X_ct).any(axis=0).sum()
if n_nan_cells > 0:
    print(f"NaN diagnostic: {n_nan_cells:,} NaN cells across "
          f"{n_nan_rows:,} rows and {n_nan_cols} cell-type columns")

# Replace NaNs with 0 (a NaN in one cell-type column for a spot just means
# that cell type was not detected there; treating it as 0 abundance is correct)
X_ct = np.nan_to_num(X_ct, nan=0.0)

# Drop only spots whose entire abundance vector is zero (these carry no info)
zero_rows = X_ct.sum(axis=1) == 0
if zero_rows.any():
    print(f"Excluding {zero_rows.sum():,} spots with zero total abundance")

valid_idx = np.where(~zero_rows)[0]
if valid_idx.size == 0:
    raise RuntimeError(
        "No spots have nonzero cell-type abundance. Check the C2L merge step."
    )
print(f"Valid spots: {valid_idx.size:,} of {adata.n_obs:,}")

X_ct_valid = X_ct[valid_idx]

# Convert counts to proportions per spot
row_sums   = X_ct_valid.sum(axis=1, keepdims=True)
X_prop     = X_ct_valid / row_sums

# Hellinger transform: sqrt of proportions, suitable for compositional data,
# stabilizes variance and makes Euclidean distance approximate Hellinger distance
X_rep = np.sqrt(X_prop, dtype=np.float32)

# Store representation back into adata for the valid spots only,
# leaving NaN rows for excluded spots so downstream code can detect them
X_full = np.full((adata.n_obs, X_rep.shape[1]), np.nan, dtype=np.float32)
X_full[valid_idx] = X_rep
adata.obsm[celltype_rep_key] = X_full
adata.uns["celltype_rep_celltypes"] = lvl4_in_key

# Cluster on the valid subset only (avoids polluting clusters with zero-vector artifacts)
adata_valid = adata[valid_idx].copy()
print(f"Clustering {adata_valid.n_obs:,} of {adata.n_obs:,} spots")

# Build neighbor graph on the Hellinger-transformed proportions
sc.pp.neighbors(
    adata_valid, use_rep=celltype_rep_key, n_neighbors=CHOSEN_K,
    key_added=neighbors_key, metric="euclidean", random_state=random_state,
)

# Leiden clustering
sc.tl.leiden(
    adata_valid, resolution=CHOSEN_RES, neighbors_key=neighbors_key,
    key_added=cluster_key, random_state=random_state,
    flavor="igraph", n_iterations=2, directed=False,
)

# UMAP for visualization only
sc.tl.umap(
    adata_valid, neighbors_key=neighbors_key,
    min_dist=umap_min_dist, spread=umap_spread, random_state=random_state,
)

# Sort cluster labels by size so cluster "0" is largest, "1" second largest, etc.
old_labels = adata_valid.obs[cluster_key].astype(str)
size_order = old_labels.value_counts().index.tolist()
relabel    = {old: str(new) for new, old in enumerate(size_order)}
new_labels = old_labels.map(relabel)
adata_valid.obs[cluster_key] = pd.Categorical(
    new_labels,
    categories=[str(i) for i in range(len(size_order))],
    ordered=False,
)

# Map cluster labels and UMAP coords back to the full adata, NaN for excluded spots
labels_full = pd.Series(pd.NA, index=adata.obs_names, dtype="string")
labels_full.iloc[valid_idx] = adata_valid.obs[cluster_key].astype(str).values
adata.obs[cluster_key] = labels_full.astype("category")

umap_full = np.full((adata.n_obs, 2), np.nan, dtype=np.float32)
umap_full[valid_idx] = adata_valid.obsm["X_umap"]
adata.obsm[umap_key] = umap_full

# Reporting
sizes = adata.obs[cluster_key].value_counts(dropna=True).sort_index()
patient_coverage = (
    adata.obs.dropna(subset=[cluster_key])
    .groupby(cluster_key, observed=True)["sample"]
    .nunique()
)
summary = pd.DataFrame({
    "n_spots": sizes,
    "n_patients": patient_coverage,
    "pct_spots": (sizes / sizes.sum() * 100).round(2),
})
print(f"\n{sizes.size} clusters from {adata.obs['sample'].nunique()} samples:")
print(summary.to_string())

# UMAP plot
sc.pl.umap(
    adata_valid, color=cluster_key,
    title=f"Cell-type niches (k={CHOSEN_K}, res={CHOSEN_RES}, n={adata_valid.n_obs:,})",
    legend_loc="on data", frameon=False, show=True,
)

del adata_valid
gc.collect()

### Save Cluster Assignments

In [ ]:
# ### Save Cluster Assignments

# %%
import pandas as pd

save_dir = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks"

cluster_key = f"celltype_leiden_k{CHOSEN_K}_r{res_str}"

umap_coords = pd.DataFrame(
    adata.obsm["X_umap_celltype"],
    index=adata.obs_names,
    columns=["UMAP1", "UMAP2"],
)
cluster_df = adata.obs[[cluster_key, "sample"]].copy()
cluster_df = cluster_df.join(umap_coords)
cluster_df.index.name = "spot_id"

out_path = f"{save_dir}/celltype_niche_clusters_k{CHOSEN_K}_r{res_str}.csv"
cluster_df.to_csv(out_path)

n_assigned = cluster_df[cluster_key].notna().sum()
print(f"Saved {cluster_df.shape[0]:,} spots ({n_assigned:,} with cluster assignments) to:")
print(out_path)
print(f"\nCluster sizes:\n{cluster_df[cluster_key].value_counts(dropna=True).sort_index().to_string()}")

### Reload Cluster Assignments from File

In [ ]:
# ### Reload Cluster Assignments from File

# %%
import numpy as np
import pandas as pd

CHOSEN_K   = 20
CHOSEN_RES = 0.6
res_str    = f"{CHOSEN_RES:g}"

save_dir     = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks"
cluster_file = f"{save_dir}/celltype_niche_clusters_k{CHOSEN_K}_r{res_str}.csv"
cluster_key  = f"celltype_leiden_k{CHOSEN_K}_r{res_str}"
umap_key     = "X_umap_celltype"

cluster_df = pd.read_csv(cluster_file, index_col=0)

# Cluster labels: stored as numeric strings, restore as category with NaN preserved
labels = cluster_df[cluster_key].astype("string")
adata.obs[cluster_key] = labels.reindex(adata.obs_names).astype("category")

# UMAP coords: NaN-safe conversion to float array
umap_df  = cluster_df[["UMAP1", "UMAP2"]].reindex(adata.obs_names)
adata.obsm[umap_key] = umap_df.to_numpy(dtype=np.float32)

n_loaded   = adata.obs[cluster_key].notna().sum()
n_clusters = adata.obs[cluster_key].cat.categories.size
print(f"Loaded {n_loaded:,} cluster assignments across {n_clusters} clusters into adata.obs['{cluster_key}']")
print(f"UMAP coords loaded into adata.obsm['{umap_key}']")
print(f"\nCluster sizes:\n{adata.obs[cluster_key].value_counts(dropna=True).sort_index().to_string()}")

### Plot Clusters on Sample

In [ ]:
# %% [markdown]
# ### Plot Clusters on Sample

# %%
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_hex

sample_id   = "CH_G_196a_vis"
cluster_key = f"celltype_leiden_k{CHOSEN_K}_r{res_str}"
spot_size   = 80
alpha       = 0.9
cmap_name   = "tab20"

ad = adata[adata.obs["sample"] == sample_id].copy()
if ad.n_obs == 0:
    raise ValueError(f"Sample '{sample_id}' not found in adata")

if "spatial" in ad.obsm:
    xy = ad.obsm["spatial"]
elif {"imagecol", "imagerow"}.issubset(ad.obs.columns):
    xy = ad.obs[["imagecol", "imagerow"]].to_numpy()
else:
    raise KeyError("Need obsm['spatial'] or obs['imagecol','imagerow'].")

# Use global category list so colors are consistent across samples
cats_global = adata.obs[cluster_key].cat.categories.tolist()
cmap        = plt.get_cmap(cmap_name, len(cats_global))
cm          = {c: to_hex(cmap(i)) for i, c in enumerate(cats_global)}

cluster_vals = ad.obs[cluster_key].astype("string")
colors = cluster_vals.map(lambda x: cm.get(x, "#cccccc") if pd.notna(x) else "#eeeeee").to_numpy()

# Sample-level cluster summary
present_cats = cluster_vals.dropna().unique().tolist()
present_cats = [c for c in cats_global if c in present_cats]
n_unassigned = cluster_vals.isna().sum()

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(xy[:, 0], xy[:, 1], c=colors, s=spot_size, edgecolors="none", alpha=alpha)
ax.set_aspect("equal")
ax.invert_yaxis()
ax.axis("off")
title = f"{sample_id} – {cluster_key} (n={ad.n_obs:,} spots"
if n_unassigned > 0:
    title += f", {n_unassigned:,} unassigned"
title += ")"
ax.set_title(title)

handles = [mpatches.Patch(color=cm[k], label=k) for k in present_cats]
if n_unassigned > 0:
    handles.append(mpatches.Patch(color="#eeeeee", label="unassigned"))
ax.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.3, 1),
          frameon=False, fontsize="small", title="Leiden cluster")
plt.tight_layout()
plt.show()

### kNN Neighborhood Analysis Based on Cell Type Cluster Assignment

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import gc

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from pandas.api.types import CategoricalDtype

# User config
cluster_key                     = f"celltype_leiden_k{CHOSEN_K}_r{CHOSEN_RES}"
nhood_key                       = "celltype_neighborhood"
coord_type                      = "grid"
n_rings                         = 1
alpha_global                    = 0.1
alpha_self                      = 1.0
pseudocount                     = 1e-6
apply_hellinger                 = True
n_pca_components                = 15
kmeans_n_clusters               = 10
kmeans_n_init                   = 25
kmeans_random_state             = 42
max_spots_per_sample_for_kmeans = None
save_dir                        = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks"

# Validate inputs
req_cols = ["sample", "imagecol", "imagerow", cluster_key]
missing  = [c for c in req_cols if c not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing required obs columns: {missing}")

if not isinstance(adata.obs[cluster_key].dtype, CategoricalDtype):
    adata.obs[cluster_key] = adata.obs[cluster_key].astype("category")

cats = adata.obs[cluster_key].cat.categories
C    = cats.size
if C < 2:
    raise ValueError(f"Need >= 2 clusters in '{cluster_key}', found {C}.")

print(f"Using AnnData with {adata.n_obs:,} spots, {C} cell-type clusters.")
print(f"Neighborhoods based on {coord_type} topology, n_rings={n_rings}.")

# Per-sample spatial neighbor composition with self + global smoothing
samples      = adata.obs["sample"].unique().tolist()
all_feats    = []
all_barcodes = []
all_samples  = []

for sid in samples:
    ad_s = adata[adata.obs["sample"] == sid].copy()
    if ad_s.n_obs == 0:
        continue

    if "spatial" not in ad_s.obsm:
        ad_s.obsm["spatial"] = ad_s.obs[["imagecol", "imagerow"]].to_numpy()

    sq.gr.spatial_neighbors(ad_s, coord_type=coord_type, n_rings=n_rings)
    A = ad_s.obsp["spatial_connectivities"].tocsr()

    codes  = ad_s.obs[cluster_key].cat.codes.to_numpy()
    X_one  = np.eye(C, dtype=np.float32)[codes]
    X_nb   = A.dot(X_one).astype(np.float32)

    X_nb  += alpha_self * X_one
    X_nb  += alpha_global * (1.0 / C)
    X_nb  += pseudocount

    row_sums  = X_nb.sum(axis=1, dtype=np.float32)
    X_nb     /= row_sums[:, None]

    all_feats.append(X_nb)
    all_barcodes.extend(ad_s.obs_names.to_list())
    all_samples.extend(ad_s.obs["sample"].astype(str).to_list())

    del ad_s, A, X_one, X_nb, row_sums, codes
    gc.collect()

if not all_feats:
    raise RuntimeError("No neighborhood features generated; check inputs and clusters.")

# Combine and optionally apply Hellinger transform
X               = np.vstack(all_feats).astype(np.float32)
all_barcodes    = np.array(all_barcodes)
all_samples_arr = np.array(all_samples)
del all_feats, all_samples
gc.collect()

if apply_hellinger:
    np.sqrt(X, out=X)

print("Neighborhood feature matrix shape:", X.shape)

# Sample-balanced subset for PCA and k-means
rng            = np.random.default_rng(kmeans_random_state)
unique_samples = np.unique(all_samples_arr)
train_indices  = []

if max_spots_per_sample_for_kmeans is None:
    train_indices = np.arange(X.shape[0], dtype=int)
else:
    for sid in unique_samples:
        idx_s = np.where(all_samples_arr == sid)[0]
        if idx_s.size == 0:
            continue
        if idx_s.size > max_spots_per_sample_for_kmeans:
            idx_s = rng.choice(idx_s, size=max_spots_per_sample_for_kmeans, replace=False)
        train_indices.append(idx_s)
    if not train_indices:
        raise RuntimeError("No training indices selected.")
    train_indices = np.concatenate(train_indices)

train_indices = np.sort(train_indices)
print(f"Training subset: {train_indices.size:,} spots ({train_indices.size / X.shape[0]:.1%} of total)")

X_train = X[train_indices]

# PCA on training subset, transform all spots
p = min(n_pca_components, X.shape[1] - 1)
pca          = PCA(n_components=p, svd_solver="randomized", random_state=kmeans_random_state)
X_train_pca  = pca.fit_transform(X_train).astype(np.float32)
X_pca        = pca.transform(X).astype(np.float32)

print(f"PCA components: {p}")
print("Explained variance (first 5 PCs):", np.round(pca.explained_variance_ratio_[:5], 4))

del X_train
gc.collect()

# k-means on training subset, predict all
km = KMeans(
    n_clusters=kmeans_n_clusters,
    n_init=kmeans_n_init,
    random_state=kmeans_random_state,
    verbose=0,
)
km.fit(X_train_pca)
labels_all = km.predict(X_pca).astype(np.int32)

del X_pca, X_train_pca
gc.collect()

print("k-means complete. Label range:", labels_all.min(), "to", labels_all.max())

# Map neighborhood labels back to full AnnData
labels_ser           = pd.Series(labels_all, index=pd.Index(all_barcodes, name="barcode"))
labels_ser           = labels_ser.astype("int32").astype("string")
mapped               = labels_ser.reindex(adata.obs_names)
adata.obs[nhood_key] = mapped.astype("category")

sizes = adata.obs[nhood_key].value_counts(dropna=True).sort_values(ascending=False)
print(f"\nNeighborhoods: {sizes.size}")
print(sizes)

if "sample" in adata.obs.columns:
    patient_coverage = (
        adata.obs.dropna(subset=[nhood_key])
        .groupby(nhood_key, observed=True)["sample"]
        .nunique()
        .sort_values(ascending=False)
    )
    print("\nPatients per neighborhood:")
    print(patient_coverage)

# Save neighborhood assignments with k value in filename
out_path = f"{save_dir}/celltype_neighborhood_annotations_k{kmeans_n_clusters}.csv"
df_save  = adata.obs[[nhood_key, "sample"]].copy()
df_save.index.name = "spot_id"
df_save.to_csv(out_path)
print(f"\nSaved {df_save.shape[0]:,} spots to {out_path}")

### Normalized Mutual Information Analysis of Cell/ECM Neighborhood Concordance

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import normalized_mutual_info_score, confusion_matrix

celltype_nhood_key = "celltype_neighborhood"
ecm_nhood_key      = "ECM_neighborhood"
ecm_csv_path       = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv"

# Load ECM neighborhood assignments and merge into adata.obs
ecm_df = pd.read_csv(ecm_csv_path, index_col=0)
print(f"ECM CSV columns: {ecm_df.columns.tolist()}")
print(f"ECM CSV index sample: {ecm_df.index[:3].tolist()}")

if ecm_nhood_key not in ecm_df.columns:
    raise KeyError(f"{ecm_nhood_key} not found in ECM CSV. Available: {ecm_df.columns.tolist()}")

adata.obs[ecm_nhood_key] = (
    ecm_df[ecm_nhood_key]
    .reindex(adata.obs_names)
    .astype("category")
)

n_matched = adata.obs[ecm_nhood_key].notna().sum()
print(f"ECM neighborhoods matched to {n_matched:,} of {adata.n_obs:,} spots")

# Keep only spots with both assignments
df = adata.obs[[celltype_nhood_key, ecm_nhood_key]].dropna().copy()
df[celltype_nhood_key] = df[celltype_nhood_key].astype(str)
df[ecm_nhood_key]      = df[ecm_nhood_key].astype(str)

print(f"Spots with both assignments: {len(df):,}")

# NMI
nmi = normalized_mutual_info_score(df[celltype_nhood_key], df[ecm_nhood_key])
print(f"Normalized Mutual Information: {nmi:.4f}  (0=no agreement, 1=perfect)")

# Row-normalized confusion matrix: rows=cell-type neighborhoods, cols=ECM neighborhoods
ct_labels  = sorted(df[celltype_nhood_key].unique(), key=lambda x: int(x))
ecm_labels = sorted(df[ecm_nhood_key].unique(), key=lambda x: int(x))

cm          = confusion_matrix(df[celltype_nhood_key], df[ecm_nhood_key], labels=ct_labels)
cm_row_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(cm_row_norm, aspect="auto", cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Proportion of cell-type neighborhood spots")

ax.set_xticks(range(len(ecm_labels)))
ax.set_xticklabels([f"ECM {l}" for l in ecm_labels], rotation=45, ha="right")
ax.set_yticks(range(len(ct_labels)))
ax.set_yticklabels([f"CT {l}" for l in ct_labels])
ax.grid(False)
ax.set_xlabel("ECM Neighborhood")
ax.set_ylabel("Cell-type Neighborhood")
ax.set_title(f"Cell-type vs ECM neighborhood overlap (NMI={nmi:.3f})")

for i in range(len(ct_labels)):
    for j in range(len(ecm_labels)):
        val = cm_row_norm[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=7, color="white" if val > 0.5 else "black")

plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

ct_cats  = sorted(adata.obs[celltype_nhood_key].dropna().astype(str).unique(), key=lambda x: int(x))
ecm_cats = sorted(adata.obs[ecm_nhood_key].dropna().astype(str).unique(), key=lambda x: int(x))

# Compute row-normalized confusion matrix per sample, then average across samples
cm_per_sample = []
for sid in adata.obs["sample"].unique():
    df_s = adata.obs.loc[
        adata.obs["sample"] == sid,
        [celltype_nhood_key, ecm_nhood_key]
    ].dropna()
    if len(df_s) < 50:
        continue
    df_s[celltype_nhood_key] = df_s[celltype_nhood_key].astype(str)
    df_s[ecm_nhood_key]      = df_s[ecm_nhood_key].astype(str)
    cm = confusion_matrix(df_s[celltype_nhood_key], df_s[ecm_nhood_key], labels=ct_cats)
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_per_sample.append(cm / row_sums)

cm_mean = np.mean(cm_per_sample, axis=0)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(cm_mean, aspect="auto", cmap="Blues", vmin=0, vmax=cm_mean.max())
plt.colorbar(im, ax=ax, label="Mean proportion across samples")

ax.set_xticks(range(len(ecm_cats)))
ax.set_xticklabels([f"ECM {l}" for l in ecm_cats], rotation=45, ha="right")
ax.set_yticks(range(len(ct_cats)))
ax.set_yticklabels([f"CT {l}" for l in ct_cats])
ax.set_xlabel("ECM Neighborhood")
ax.set_ylabel("Cell-type Neighborhood")
ax.set_title(f"Cell-type vs ECM neighborhood overlap (sample-weighted, n={len(cm_per_sample)} samples)")
ax.grid(False)

for i in range(len(ct_cats)):
    for j in range(len(ecm_cats)):
        val = cm_mean[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=7, color="white" if val > cm_mean.max() * 0.6 else "black")

plt.tight_layout()
plt.show()

### Load Saved ECM Neigborhood Annotations

In [ ]:
import pandas as pd

ecm_csv_path = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv"
ecm_annot    = pd.read_csv(ecm_csv_path).set_index("spot_id")
adata.obs["ECM_neighborhood"] = (
    ecm_annot["ECM_neighborhood"]
    .reindex(adata.obs_names)
    .astype("category")
)

### Plot Cell Type Neighborhoods

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import matplotlib.patches as mpatches

sample_id    = "CH_G_196a_vis"
spot_size    = 80
alpha        = 0.9
cmap_name    = "tab20"
ecm_csv_path = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv"

# Load ECM neighborhoods if not already in adata.obs
if "ECM_neighborhood" not in adata.obs.columns:
    ecm_annot = pd.read_csv(ecm_csv_path).set_index("spot_id")
    adata.obs["ECM_neighborhood"] = (
        ecm_annot["ECM_neighborhood"]
        .reindex(adata.obs_names)
        .astype("category")
    )
    print(f"Loaded ECM neighborhoods: {adata.obs['ECM_neighborhood'].notna().sum():,} spots matched")

ad = adata[adata.obs["sample"] == sample_id].copy()

if "spatial" in ad.obsm:
    xy = ad.obsm["spatial"]
elif {"imagecol", "imagerow"}.issubset(ad.obs.columns):
    xy = ad.obs[["imagecol", "imagerow"]].to_numpy()
else:
    raise KeyError("Need obsm['spatial'] or obs['imagecol','imagerow'].")

def make_color_map(series, cmap_name):
    cats   = sorted(series.dropna().astype(str).unique(), key=lambda x: int(x))
    cmap   = plt.get_cmap(cmap_name, len(cats))
    colors = [to_hex(cmap(i)) for i in range(len(cats))]
    return cats, dict(zip(cats, colors))

ct_cats,  ct_cm  = make_color_map(ad.obs["celltype_neighborhood"], cmap_name)
ecm_cats, ecm_cm = make_color_map(ad.obs["ECM_neighborhood"], cmap_name)

ct_colors  = ad.obs["celltype_neighborhood"].astype(str).map(lambda x: ct_cm.get(x, "#999999")).to_numpy()
ecm_colors = ad.obs["ECM_neighborhood"].astype(str).map(lambda x: ecm_cm.get(x, "#999999")).to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

for ax, colors, cats, cm, title in [
    (axes[0], ct_colors,  ct_cats,  ct_cm,  "Cell-type neighborhood"),
    (axes[1], ecm_colors, ecm_cats, ecm_cm, "ECM neighborhood"),
]:
    ax.scatter(xy[:, 0], xy[:, 1], c=colors, s=spot_size, edgecolors="none", alpha=alpha)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title(f"{sample_id} – {title}")
    handles = [mpatches.Patch(color=cm[k], label=k) for k in cats if k in cm]
    ax.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.3, 1),
              frameon=False, fontsize="small", title="Neighborhood")

plt.tight_layout()
plt.show()

### Plot Cell Type Neighborhoods Across Cohort

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.colors import to_hex

nhood_key             = "celltype_neighborhood"
sample_key            = "sample"
normalize             = True
top_n_neighborhoods   = None
other_label           = "Other"
min_spots_per_sample  = 200
cluster_samples       = True
cluster_neighborhoods = True
fig_width, fig_height = 18, 8
legend_outside        = True
legend_ncol           = 2
cmap_name             = "tab20"

for c in (sample_key, nhood_key):
    if c not in adata.obs.columns:
        raise KeyError(f"{c} not in adata.obs")

if not adata.obs[nhood_key].dtype.name == "category":
    adata.obs[nhood_key] = adata.obs[nhood_key].astype("category")

df    = adata.obs[[sample_key, nhood_key]].dropna()
pivot = df.groupby([sample_key, nhood_key], observed=True).size().unstack(fill_value=0)
pivot = pivot.loc[pivot.sum(axis=1) >= min_spots_per_sample]
if pivot.empty:
    raise ValueError("No samples left after min_spots_per_sample filter.")

if top_n_neighborhoods is not None and pivot.shape[1] > top_n_neighborhoods:
    totals = pivot.sum(axis=0).sort_values(ascending=False)
    keep   = totals.index[:top_n_neighborhoods]
    other  = pivot.drop(columns=keep).sum(axis=1)
    pivot  = pivot[keep]
    pivot[other_label] = other

if normalize:
    pivot = pivot.div(pivot.sum(axis=1), axis=0)

if cluster_neighborhoods and pivot.shape[1] > 2:
    Zc        = linkage(pivot.T.values.astype(np.float32), method="average", metric="cosine")
    order_cols = dendrogram(Zc, no_plot=True)["leaves"]
    pivot     = pivot.iloc[:, order_cols]
else:
    pivot = pivot.loc[:, pivot.sum(0).sort_values(ascending=False).index]

if cluster_samples and pivot.shape[0] > 2:
    Zs        = linkage(pivot.values.astype(np.float32), method="average", metric="cosine")
    order_rows = dendrogram(Zs, no_plot=True)["leaves"]
    pivot     = pivot.iloc[order_rows]

# Use colors consistent with the spatial plots
cats_key   = f"{nhood_key}_categories"
colors_key = f"{nhood_key}_colors"

if cats_key in adata.uns and colors_key in adata.uns:
    cats_global   = list(adata.uns[cats_key])
    colors_global = list(adata.uns[colors_key])
    cm = dict(zip(cats_global, colors_global))
else:
    cats_global   = adata.obs[nhood_key].cat.categories.astype(str).tolist()
    cmap          = plt.get_cmap(cmap_name, len(cats_global))
    colors_global = [to_hex(cmap(i)) for i in range(len(cats_global))]
    adata.uns[cats_key]   = cats_global
    adata.uns[colors_key] = colors_global
    cm = dict(zip(cats_global, colors_global))

colors = [
    "#b0b0b0" if col == other_label else cm.get(str(col), "#999999")
    for col in pivot.columns
]

fig, ax = plt.subplots(figsize=(fig_width, fig_height))
x      = np.arange(pivot.shape[0])
bottom = np.zeros(pivot.shape[0], dtype=float)

for col, color in zip(pivot.columns, colors):
    vals = pivot[col].to_numpy()
    ax.bar(x, vals, bottom=bottom, width=0.9, color=color, edgecolor="none", label=str(col))
    bottom += vals

ax.set_xlim(-0.5, len(x) - 0.5)
ax.set_ylabel("Proportion" if normalize else "Spot count")
ax.set_title("Cell-type neighborhoods across patients")

step = max(1, len(x) // 40)
ax.set_xticks(x[::step])
ax.set_xticklabels(pivot.index[::step], rotation=90, fontsize=8)
ax.margins(x=0)
ax.grid(False)

if legend_outside:
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5),
              frameon=False, ncol=legend_ncol, fontsize=8)
else:
    ax.legend(frameon=False, ncol=legend_ncol, fontsize=8)

plt.tight_layout()
plt.show()

pivot.to_csv("celltype_neighborhoods_by_patient_stacked_matrix.csv")
print(f"Saved matrix with shape {pivot.shape} to 'celltype_neighborhoods_by_patient_stacked_matrix.csv'")

### Cell Type Composition of Cell Neighborhoods

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list
from matplotlib.colors import to_hex

nhood_key        = "celltype_neighborhood"
c2l_prefix       = "meanscell_abundance_w_sf_"
curation_col     = "celltype_level4_curated"
top_n            = 15
figsize          = (16, 6)
bar_width        = 0.75
other_color      = "#ebecf0"
legend_font      = 8
legend_ncol      = 1
EXCLUDE_PATTERNS = ["doublet", "other"]

if nhood_key not in adata.obs.columns:
    raise ValueError(f"{nhood_key} not found in adata.obs")
if "celltype_annotation_key" not in adata.uns:
    raise ValueError("adata.uns['celltype_annotation_key'] not found.")

celltype_key = adata.uns["celltype_annotation_key"]

c2l_cols   = [c for c in adata.obs.columns if c.startswith(c2l_prefix)]
lvl4_names = pd.Index([c.replace(c2l_prefix, "") for c in c2l_cols])
counts     = adata.obs[c2l_cols].copy()
counts.columns = lvl4_names

lvl4_in_key  = lvl4_names.intersection(celltype_key.index)
counts       = counts[lvl4_in_key]
curated_map  = celltype_key.loc[lvl4_in_key, curation_col].astype(str).fillna("Unknown")

keep_mask   = ~curated_map.str.lower().apply(
    lambda x: any(pat in x for pat in EXCLUDE_PATTERNS)
)
counts      = counts.loc[:, keep_mask.values]
curated_map = curated_map[keep_mask]

print(f"Excluded cell types: {(~keep_mask).sum()}, Retained: {keep_mask.sum()}")

counts_curated = counts.T.groupby(curated_map).sum().T

row_sums = counts_curated.sum(axis=1).replace(0, np.nan)
frac     = counts_curated.div(row_sums, axis=0).fillna(0.0)

nh    = adata.obs[nhood_key].astype("category")
means = (
    frac.groupby(nh)
        .mean()
        .pipe(lambda df: df.div(df.sum(axis=1), axis=0))
        .fillna(0.0)
)

H         = np.sqrt(means.values.astype(np.float64))
Z         = linkage(H, method="average", metric="euclidean")
order_idx = leaves_list(Z)
nhood_order = [means.index[i] for i in order_idx]

top_map = {nh_name: row.sort_values(ascending=False).head(top_n)
           for nh_name, row in means.iterrows()}

union_types       = sorted(set().union(*[s.index for s in top_map.values()]))
global_importance = pd.Series(0.0, index=union_types)
for s in top_map.values():
    global_importance.loc[s.index] += s.values
union_types_sorted = list(global_importance.sort_values(ascending=False).index)

tab20   = list(plt.get_cmap("tab20").colors) + list(plt.get_cmap("tab20b").colors)
palette = {ct: to_hex(tab20[i % len(tab20)]) for i, ct in enumerate(union_types_sorted)}

x                = np.arange(len(nhood_order))
heights_by_type  = {
    ct: np.array([float(top_map[nh].get(ct, 0.0)) for nh in nhood_order])
    for ct in union_types_sorted
}
other_heights = np.array(
    [max(0.0, 1.0 - float(top_map[nh].sum())) for nh in nhood_order]
)

fig, ax = plt.subplots(figsize=figsize)
bottom  = np.zeros(len(nhood_order), dtype=float)

for ct in union_types_sorted:
    h = heights_by_type[ct]
    if np.all(h == 0):
        continue
    ax.bar(x, h, bottom=bottom, width=bar_width, label=ct,
           color=palette[ct], edgecolor="white", linewidth=0.3)
    bottom += h

if not np.all(other_heights == 0):
    ax.bar(x, other_heights, bottom=bottom, width=bar_width, label="Other",
           color=other_color, edgecolor="white", linewidth=0.3)

ax.set_xticks(x)
ax.set_xticklabels([f"N{nh}" for nh in nhood_order], fontsize=11, fontweight="bold")
ax.set_ylabel("Proportion", fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_xlim(-0.5, len(x) - 0.5)
ax.margins(x=0.01)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_title("Cell-type composition per cell-type neighborhood", fontsize=13, fontweight="bold")

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5),
          frameon=False, fontsize=legend_font, ncol=legend_ncol,
          title="Cell type", title_fontsize=9, handlelength=1.2,
          handleheight=1.0, labelspacing=0.4)

plt.tight_layout()
plt.savefig("celltype_neighborhood_cell_landscape.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: celltype_neighborhood_cell_landscape.png")

In [ ]:
cluster_key = "celltype_leiden_k20_r0.5"
c2l_prefix  = "meanscell_abundance_w_sf_"
curation_col = "celltype_level4_curated"
celltype_key = adata.uns["celltype_annotation_key"]

c2l_cols   = [c for c in adata.obs.columns if c.startswith(c2l_prefix)]
lvl4_names = pd.Index([c.replace(c2l_prefix, "") for c in c2l_cols])
counts     = adata.obs[c2l_cols].copy()
counts.columns = lvl4_names

lvl4_in_key    = lvl4_names.intersection(celltype_key.index)
counts         = counts[lvl4_in_key]
curated_map    = celltype_key.loc[lvl4_in_key, curation_col].astype(str)
counts_curated = counts.T.groupby(curated_map).sum().T
row_sums       = counts_curated.sum(axis=1).replace(0, np.nan)
frac           = counts_curated.div(row_sums, axis=0).fillna(0.0)

for cl in ["11", "12", "13", "19", "35"]:
    mask  = adata.obs[cluster_key].astype(str) == cl
    top5  = frac[mask].mean().sort_values(ascending=False).head(5)
    print(f"\nLeiden cluster {cl} (n={mask.sum():,} spots):")
    print(top5.round(4))

# Second Strategy - Build kNN Neighborhoods Directly from Cell Comp

In [ ]:
# ### Spatial Neighborhood Analysis from Cell Type Composition
#
# Build neighborhood vectors by averaging the Hellinger-transformed cell type
# proportion vectors of each spot and its spatial neighbors. Cluster these
# vectors with kmeans to define spatial niches that are directly interpretable
# as average cell type compositions.

# %%
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import gc

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# User config
celltype_rep_key                = "X_celltype"
nhood_key                       = "celltype_neighborhood"
coord_type                      = "grid"
n_rings                         = 1
include_self                    = True
kmeans_n_clusters               = 10
kmeans_n_init                   = 25
kmeans_random_state             = 42
max_spots_per_sample_for_kmeans = None
save_dir                        = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks"

# Validate inputs
req_cols = ["sample", "imagecol", "imagerow"]
missing  = [c for c in req_cols if c not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing required obs columns: {missing}")
if celltype_rep_key not in adata.obsm:
    raise ValueError(
        f"adata.obsm['{celltype_rep_key}'] not found. "
        f"Run the cell type clustering cell first to generate it."
    )

X_rep = adata.obsm[celltype_rep_key]
n_features = X_rep.shape[1]
print(f"Cell type representation: {n_features} cell types, {X_rep.shape[0]:,} spots")

# Per-sample spatial neighborhood averaging
# For each spot, the neighborhood vector is the mean Hellinger-proportion
# vector of its spatial neighbors (and itself, if include_self=True)
samples       = adata.obs["sample"].unique().tolist()
all_feats     = []
all_barcodes  = []
all_samples   = []

for sid in samples:
    mask = (adata.obs["sample"] == sid).to_numpy()
    if not mask.any():
        continue

    ad_s = adata[mask].copy()

    # Drop spots with NaN in the representation (came from invalid C2L spots)
    valid = ~np.isnan(ad_s.obsm[celltype_rep_key]).any(axis=1)
    if valid.sum() < 10:
        continue
    ad_s = ad_s[valid].copy()

    if "spatial" not in ad_s.obsm:
        ad_s.obsm["spatial"] = ad_s.obs[["imagecol", "imagerow"]].to_numpy()

    # Build spatial neighbor graph (Visium hex grid topology)
    sq.gr.spatial_neighbors(ad_s, coord_type=coord_type, n_rings=n_rings)
    A = ad_s.obsp["spatial_connectivities"].tocsr().astype(np.float32)

    # Optionally include each spot itself in its own neighborhood
    if include_self:
        from scipy.sparse import eye as speye
        A = A + speye(A.shape[0], dtype=np.float32, format="csr")

    # Average the cell type representation across spatial neighbors
    X_self  = ad_s.obsm[celltype_rep_key].astype(np.float32)
    X_sum   = A.dot(X_self)
    n_neigh = np.asarray(A.sum(axis=1)).ravel()
    n_neigh[n_neigh == 0] = 1.0
    X_nb    = X_sum / n_neigh[:, None]

    all_feats.append(X_nb)
    all_barcodes.extend(ad_s.obs_names.tolist())
    all_samples.extend([sid] * ad_s.n_obs)

    del ad_s, A, X_self, X_sum, X_nb, n_neigh
    gc.collect()

if not all_feats:
    raise RuntimeError("No neighborhood features generated; check inputs.")

X_nbhd          = np.vstack(all_feats).astype(np.float32)
all_barcodes    = np.array(all_barcodes)
all_samples_arr = np.array(all_samples)
del all_feats, all_samples
gc.collect()

print(f"Neighborhood feature matrix: {X_nbhd.shape}")

# Sample-balanced subset for kmeans training (optional)
rng = np.random.default_rng(kmeans_random_state)
if max_spots_per_sample_for_kmeans is None:
    train_indices = np.arange(X_nbhd.shape[0], dtype=int)
else:
    train_indices = []
    for sid in np.unique(all_samples_arr):
        idx_s = np.where(all_samples_arr == sid)[0]
        if idx_s.size > max_spots_per_sample_for_kmeans:
            idx_s = rng.choice(idx_s, size=max_spots_per_sample_for_kmeans, replace=False)
        train_indices.append(idx_s)
    train_indices = np.sort(np.concatenate(train_indices))

print(f"kmeans training subset: {train_indices.size:,} spots "
      f"({train_indices.size / X_nbhd.shape[0]:.1%} of total)")

# kmeans on neighborhood vectors directly (no PCA — interpretability matters
# more than dimensionality here, and 72 features is not large for kmeans)
km = KMeans(
    n_clusters=kmeans_n_clusters,
    n_init=kmeans_n_init,
    random_state=kmeans_random_state,
    verbose=0,
)
km.fit(X_nbhd[train_indices])
labels_all = km.predict(X_nbhd).astype(np.int32)

# Sort neighborhoods by size so neighborhood "0" is the most prevalent
sizes_raw = pd.Series(labels_all).value_counts().sort_values(ascending=False)
relabel   = {int(old): str(new) for new, old in enumerate(sizes_raw.index)}
labels_str = pd.Series(labels_all).map(relabel).to_numpy()

print(f"\nkmeans complete: {kmeans_n_clusters} neighborhoods")

# Map labels back to full adata
labels_ser           = pd.Series(labels_str, index=pd.Index(all_barcodes, name="barcode"))
adata.obs[nhood_key] = labels_ser.reindex(adata.obs_names).astype("category")

# Reporting
sizes = adata.obs[nhood_key].value_counts(dropna=True).sort_index()
patient_coverage = (
    adata.obs.dropna(subset=[nhood_key])
    .groupby(nhood_key, observed=True)["sample"]
    .nunique()
)
summary = pd.DataFrame({
    "n_spots": sizes,
    "n_patients": patient_coverage,
    "pct_spots": (sizes / sizes.sum() * 100).round(2),
})
print(f"\n{sizes.size} neighborhoods from {adata.obs['sample'].nunique()} samples:")
print(summary.to_string())

# Save the kmeans centroids (in Hellinger-proportion space) so the neighborhood
# composition readout is directly available
celltype_names = adata.uns.get("celltype_rep_celltypes", [f"ct_{i}" for i in range(n_features)])
centroid_df = pd.DataFrame(
    km.cluster_centers_,
    columns=celltype_names,
)
# Reindex centroids to match the size-sorted label order
inv_relabel = {v: k for k, v in relabel.items()}
centroid_df = centroid_df.iloc[[inv_relabel[str(i)] for i in range(kmeans_n_clusters)]].reset_index(drop=True)
centroid_df.index = [str(i) for i in range(kmeans_n_clusters)]
centroid_df.index.name = nhood_key
adata.uns[f"{nhood_key}_centroids_hellinger"] = centroid_df

# Convert centroids back to proportion space (squared) for direct interpretation
centroid_prop = centroid_df ** 2
centroid_prop = centroid_prop.div(centroid_prop.sum(axis=1), axis=0)
adata.uns[f"{nhood_key}_centroids_proportion"] = centroid_prop

# Save neighborhood assignments to disk
out_path = f"{save_dir}/celltype_neighborhood_annotations_k{kmeans_n_clusters}.csv"
df_save  = adata.obs[[nhood_key, "sample"]].copy()
df_save.index.name = "spot_id"
df_save.to_csv(out_path)
print(f"\nSaved {df_save.shape[0]:,} spots to {out_path}")

# Save centroids (proportion space) for cross-session reference
centroid_path = f"{save_dir}/celltype_neighborhood_centroids_k{kmeans_n_clusters}.csv"
centroid_prop.to_csv(centroid_path)
print(f"Saved neighborhood centroids (cell type proportions) to {centroid_path}")

### Plot CT Neighborhoods on Sample

In [ ]:
# ### Plot Cell Type and ECM Neighborhoods Side-by-Side

# %%
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_hex

# User config
sample_id    = "CH_G_196a_vis"
ct_nhood_key = "celltype_neighborhood"
ecm_nhood_key = "ECM_neighborhood"
spot_size    = 80
alpha        = 0.9
ct_cmap_name  = "tab20"
ecm_cmap_name = "tab20b"
unassigned_color = "#e8e8e8"
ecm_csv_path = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv"

# Load ECM neighborhoods if not already in adata.obs
if ecm_nhood_key not in adata.obs.columns:
    ecm_annot = pd.read_csv(ecm_csv_path).set_index("spot_id")
    adata.obs[ecm_nhood_key] = (
        ecm_annot[ecm_nhood_key]
        .reindex(adata.obs_names)
        .astype("category")
    )

# Subset to the sample
ad = adata[adata.obs["sample"] == sample_id].copy()
if ad.n_obs == 0:
    raise ValueError(f"Sample '{sample_id}' not found in adata")

if "spatial" in ad.obsm:
    xy = ad.obsm["spatial"]
elif {"imagecol", "imagerow"}.issubset(ad.obs.columns):
    xy = ad.obs[["imagecol", "imagerow"]].to_numpy()
else:
    raise KeyError("Need obsm['spatial'] or obs['imagecol','imagerow'].")

# Build a label-to-dominant-celltype map for the cell type neighborhoods
# so the legend is biologically informative
centroid_key = f"{ct_nhood_key}_centroids_proportion"
nh_label_map = {}
if centroid_key in adata.uns:
    centroids = adata.uns[centroid_key]
    for nh in centroids.index:
        top_ct = centroids.loc[nh].sort_values(ascending=False).index[0]
        # Shorten very long cell type names for legend readability
        short = top_ct.replace("_", " ")
        if len(short) > 25:
            short = short[:22] + "…"
        nh_label_map[nh] = f"{nh}: {short}"
else:
    nh_label_map = {}

def make_color_map(series, cmap_name):
    cats = sorted(series.dropna().astype(str).unique(), key=lambda x: int(x))
    cmap = plt.get_cmap(cmap_name, max(len(cats), 1))
    return cats, {c: to_hex(cmap(i)) for i, c in enumerate(cats)}

# Use global category lists (across cohort), not just the current sample,
# so colors stay consistent between samples
def global_categories(key, cmap_name):
    if key not in adata.obs.columns:
        return [], {}
    series = adata.obs[key].astype("string").dropna()
    cats = sorted(series.unique(), key=lambda x: int(x))
    cmap = plt.get_cmap(cmap_name, max(len(cats), 1))
    return cats, {c: to_hex(cmap(i)) for i, c in enumerate(cats)}

ct_cats_all,  ct_cm  = global_categories(ct_nhood_key,  ct_cmap_name)
ecm_cats_all, ecm_cm = global_categories(ecm_nhood_key, ecm_cmap_name)

def color_array(series, color_map):
    s = series.astype("string")
    return s.map(lambda x: color_map.get(x, unassigned_color) if pd.notna(x) else unassigned_color).to_numpy()

ct_colors  = color_array(ad.obs[ct_nhood_key],  ct_cm)
ecm_colors = color_array(ad.obs[ecm_nhood_key], ecm_cm)

# Restrict legend to neighborhoods actually present in this sample
ct_present  = [c for c in ct_cats_all  if c in ad.obs[ct_nhood_key].astype("string").dropna().unique()]
ecm_present = [c for c in ecm_cats_all if c in ad.obs[ecm_nhood_key].astype("string").dropna().unique()]

n_ct_unassigned  = ad.obs[ct_nhood_key].isna().sum()
n_ecm_unassigned = ad.obs[ecm_nhood_key].isna().sum()

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

panels = [
    (axes[0], ct_colors,  ct_present,  ct_cm,  "Cell-type neighborhood",  n_ct_unassigned, nh_label_map),
    (axes[1], ecm_colors, ecm_present, ecm_cm, "ECM neighborhood",        n_ecm_unassigned, {}),
]

for ax, colors, present, cm, title, n_unassigned, label_map in panels:
    ax.scatter(xy[:, 0], xy[:, 1], c=colors, s=spot_size, edgecolors="none", alpha=alpha)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    panel_title = f"{sample_id}\n{title}"
    if n_unassigned > 0:
        panel_title += f"  ({n_unassigned:,} unassigned)"
    ax.set_title(panel_title, fontsize=12, fontweight="semibold", loc="left")

    handles = [mpatches.Patch(color=cm[k], label=label_map.get(k, k)) for k in present]
    if n_unassigned > 0:
        handles.append(mpatches.Patch(color=unassigned_color, label="unassigned"))
    ax.legend(
        handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0),
        frameon=False, fontsize=9, title="Niche", title_fontsize=10,
        handlelength=1.0, handleheight=1.0, labelspacing=0.4,
    )

plt.tight_layout()
plt.show()

### NMI Comparison between CT and ECM Neighborhoods

In [ ]:
# %% [markdown]
# ### Concordance of Cell Type and ECM Neighborhoods
#
# Quantify how cell-type and ECM spatial niches overlap. NMI is computed
# both globally and per-sample so between-patient heterogeneity in the
# cell-ECM coupling is visible. The joint frequency matrix shows which
# niche pairs co-occur, with marginals showing each neighborhood's prevalence.

# %%
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
)

# User config
ct_nhood_key  = "celltype_neighborhood"
ecm_nhood_key = "ECM_neighborhood"
ecm_csv_path  = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv"
min_spots_per_sample = 100

# Load ECM neighborhoods if not already in adata.obs
if ecm_nhood_key not in adata.obs.columns:
    ecm_annot = pd.read_csv(ecm_csv_path).set_index("spot_id")
    adata.obs[ecm_nhood_key] = (
        ecm_annot[ecm_nhood_key]
        .reindex(adata.obs_names)
        .astype("category")
    )

# Spots with both assignments
df = adata.obs[[ct_nhood_key, ecm_nhood_key, "sample"]].dropna(
    subset=[ct_nhood_key, ecm_nhood_key]
).copy()
df[ct_nhood_key]  = df[ct_nhood_key].astype(str)
df[ecm_nhood_key] = df[ecm_nhood_key].astype(str)
print(f"Spots with both assignments: {len(df):,} from {df['sample'].nunique()} samples")

# Global metrics
nmi_global = normalized_mutual_info_score(df[ct_nhood_key], df[ecm_nhood_key])
ari_global = adjusted_rand_score(df[ct_nhood_key], df[ecm_nhood_key])
print(f"\nGlobal NMI: {nmi_global:.3f}   Global ARI: {ari_global:.3f}")
print("(NMI: 0 = independent, 1 = identical. "
      "ARI: 0 = chance, 1 = identical, can be negative)")

# Per-sample metrics
per_sample_records = []
for sid, dsub in df.groupby("sample"):
    if len(dsub) < min_spots_per_sample:
        continue
    if dsub[ct_nhood_key].nunique() < 2 or dsub[ecm_nhood_key].nunique() < 2:
        continue
    per_sample_records.append({
        "sample": sid,
        "n_spots": len(dsub),
        "nmi": normalized_mutual_info_score(dsub[ct_nhood_key], dsub[ecm_nhood_key]),
        "ari": adjusted_rand_score(dsub[ct_nhood_key], dsub[ecm_nhood_key]),
    })
per_sample = pd.DataFrame(per_sample_records)
print(f"\nPer-sample (n={len(per_sample)} samples):")
print(f"  NMI: median={per_sample['nmi'].median():.3f}, "
      f"IQR=[{per_sample['nmi'].quantile(0.25):.3f}, {per_sample['nmi'].quantile(0.75):.3f}]")
print(f"  ARI: median={per_sample['ari'].median():.3f}, "
      f"IQR=[{per_sample['ari'].quantile(0.25):.3f}, {per_sample['ari'].quantile(0.75):.3f}]")

# Joint frequency matrix (sample-averaged) in PERCENT units throughout
ct_labels  = sorted(df[ct_nhood_key].unique(),  key=lambda x: int(x))
ecm_labels = sorted(df[ecm_nhood_key].unique(), key=lambda x: int(x))

def joint_freq_pct(dsub, ct_labels, ecm_labels):
    cm = pd.crosstab(
        dsub[ct_nhood_key], dsub[ecm_nhood_key], dropna=False,
    ).reindex(index=ct_labels, columns=ecm_labels, fill_value=0).to_numpy(dtype=float)
    total = cm.sum()
    return (cm / total * 100.0) if total > 0 else cm

joint_per_sample = []
for sid, dsub in df.groupby("sample"):
    if len(dsub) < min_spots_per_sample:
        continue
    joint_per_sample.append(joint_freq_pct(dsub, ct_labels, ecm_labels))

joint_mean   = np.mean(joint_per_sample, axis=0)        # in percent, sums to 100
ct_marginal  = joint_mean.sum(axis=1)                   # CT neighborhood prevalence (%)
ecm_marginal = joint_mean.sum(axis=0)                   # ECM neighborhood prevalence (%)

# ----- Figure 1: Joint frequency matrix with marginals -----
heatmap_cmap = LinearSegmentedColormap.from_list(
    "ct_ecm", ["#ffffff", "#d4ecf2", "#7ec1cf", "#2c7a8a", "#0d3a4a"], N=256,
)
bar_color = "#7ec1cf"

fig = plt.figure(figsize=(11, 9))
gs = GridSpec(
    2, 3,
    width_ratios=[6, 1.4, 0.25],
    height_ratios=[1.0, 6],
    wspace=0.08, hspace=0.08,
    left=0.10, right=0.93, top=0.92, bottom=0.10,
)

ax_top  = fig.add_subplot(gs[0, 0])
ax_main = fig.add_subplot(gs[1, 0])
ax_right = fig.add_subplot(gs[1, 1])
ax_cbar  = fig.add_subplot(gs[1, 2])

# Main heatmap
im = ax_main.imshow(joint_mean, aspect="auto", cmap=heatmap_cmap,
                     vmin=0, vmax=joint_mean.max())
ax_main.set_xticks(range(len(ecm_labels)))
ax_main.set_xticklabels([f"E{l}" for l in ecm_labels], fontsize=10)
ax_main.set_yticks(range(len(ct_labels)))
ax_main.set_yticklabels([f"CT{l}" for l in ct_labels], fontsize=10)
ax_main.set_xlabel("ECM neighborhood", fontsize=11)
ax_main.set_ylabel("Cell-type neighborhood", fontsize=11)
ax_main.tick_params(length=0)
ax_main.grid(False)
for spine in ax_main.spines.values():
    spine.set_visible(False)

# Cell annotations: only label cells above a small threshold
threshold = max(joint_mean.max() * 0.05, 0.5)
for i in range(len(ct_labels)):
    for j in range(len(ecm_labels)):
        v = joint_mean[i, j]
        if v >= threshold:
            ax_main.text(j, i, f"{v:.1f}", ha="center", va="center",
                         fontsize=8,
                         color="white" if v > joint_mean.max() * 0.55 else "#222222")

# Top marginal: ECM neighborhood prevalence
ax_top.bar(range(len(ecm_labels)), ecm_marginal,
           color=bar_color, edgecolor="none", width=0.85)
ax_top.set_xlim(ax_main.get_xlim())
ax_top.set_xticks([])
ax_top.set_ylabel("ECM\n%", fontsize=9, rotation=0, ha="right", va="center", labelpad=12)
ax_top.tick_params(axis="y", labelsize=8)
ax_top.grid(False)
for s in ["top", "right", "bottom"]:
    ax_top.spines[s].set_visible(False)

# Right marginal: CT neighborhood prevalence
ax_right.barh(range(len(ct_labels)), ct_marginal,
              color=bar_color, edgecolor="none", height=0.85)
ax_right.set_ylim(ax_main.get_ylim())
ax_right.set_yticks([])
ax_right.set_xlabel("CT %", fontsize=9)
ax_right.tick_params(axis="x", labelsize=8)
ax_right.grid(False)
for s in ["top", "right", "left"]:
    ax_right.spines[s].set_visible(False)

# Colorbar
cbar = plt.colorbar(im, cax=ax_cbar)
cbar.set_label("Joint frequency (%)", fontsize=10)
cbar.ax.tick_params(labelsize=8)

fig.suptitle(
    f"Cell-type vs ECM neighborhood concordance — joint frequency\n"
    f"global NMI={nmi_global:.3f}, ARI={ari_global:.3f}, n={len(per_sample)} samples",
    fontsize=12, fontweight="semibold",
)
plt.show()

# ----- Figure 2: Per-sample NMI and ARI distributions -----
fig2, (ax_nmi, ax_ari) = plt.subplots(1, 2, figsize=(11, 4))

for ax, col, color, label in [
    (ax_nmi, "nmi", "#2c7a8a", "NMI"),
    (ax_ari, "ari", "#a85a3c", "ARI"),
]:
    vals = per_sample[col]
    ax.hist(vals, bins=25, color=color, edgecolor="white", linewidth=0.5, alpha=0.85)
    global_val = nmi_global if col == "nmi" else ari_global
    ax.axvline(global_val, color="#c0392b", linestyle="--", linewidth=1.5,
               label=f"Global = {global_val:.3f}")
    ax.axvline(vals.median(), color="#222222", linestyle=":", linewidth=1.5,
               label=f"Median = {vals.median():.3f}")
    ax.set_xlabel(f"Per-sample {label}", fontsize=10)
    ax.set_ylabel("Number of samples", fontsize=10)
    ax.legend(loc="upper right", fontsize=9, frameon=False)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize=9)

fig2.suptitle("Per-sample concordance distribution",
              fontsize=12, fontweight="semibold", y=1.02)
plt.tight_layout()
plt.show()

# Store results
adata.uns["ct_ecm_concordance"] = {
    "nmi_global": nmi_global,
    "ari_global": ari_global,
    "per_sample": per_sample,
    "joint_mean_pct": pd.DataFrame(
        joint_mean,
        index=[f"CT{l}" for l in ct_labels],
        columns=[f"E{l}" for l in ecm_labels],
    ),
    "ct_marginal_pct":  pd.Series(ct_marginal,  index=[f"CT{l}" for l in ct_labels]),
    "ecm_marginal_pct": pd.Series(ecm_marginal, index=[f"E{l}"  for l in ecm_labels]),
}
print("\nResults stored in adata.uns['ct_ecm_concordance']")

### Dotplot of Cell Compoistion per Neighborhood

In [ ]:
# %% [markdown]
# ### Cell-type marker dotplot — all cell types
#
# Marker score (composition × log2 FC⁺) per niche. Color = marker score.
# Size = specificity (fraction of cell type's total cohort abundance in
# this niche). Bright + large = strong niche-defining marker.

# %%
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list

# User config
nhood_key        = "celltype_neighborhood"
c2l_prefix       = "meanscell_abundance_w_sf_"
curation_col     = "celltype_level4_curated"
EXCLUDE_PATTERNS = ["doublet", "other", "unknown"]

# Sanity checks
if nhood_key not in adata.obs.columns:
    raise ValueError(f"{nhood_key} not found in adata.obs")
if "celltype_annotation_key" not in adata.uns:
    raise ValueError("adata.uns['celltype_annotation_key'] not found.")

celltype_key = adata.uns["celltype_annotation_key"]

# Pull abundance and translate level-4 → curated names
c2l_cols   = [c for c in adata.obs.columns if c.startswith(c2l_prefix)]
lvl4_names = pd.Index([c.replace(c2l_prefix, "") for c in c2l_cols])
counts     = adata.obs[c2l_cols].copy()
counts.columns = lvl4_names

lvl4_in_key  = lvl4_names.intersection(celltype_key.index)
counts       = counts[lvl4_in_key]
curated_map  = celltype_key.loc[lvl4_in_key, curation_col].astype(str).fillna("Unknown")

keep_mask = ~curated_map.str.lower().apply(
    lambda x: any(pat in x for pat in EXCLUDE_PATTERNS)
)
counts      = counts.loc[:, keep_mask.values]
curated_map = curated_map[keep_mask]

counts_curated = counts.T.groupby(curated_map).sum().T

# Spots with valid neighborhood and abundance
nh = adata.obs[nhood_key].astype("string")
valid_mask = nh.notna() & (counts_curated.sum(axis=1) > 0)
counts_curated = counts_curated.loc[valid_mask]
nh_valid = nh.loc[valid_mask]

# Core matrices
abund_per_nh = counts_curated.groupby(nh_valid, observed=True).sum()
composition  = abund_per_nh.div(abund_per_nh.sum(axis=1), axis=0).fillna(0)
specificity  = abund_per_nh.div(abund_per_nh.sum(axis=0), axis=1).fillna(0)
ct_global    = abund_per_nh.sum(axis=0) / abund_per_nh.values.sum()
eps          = 1e-6
log2fc       = np.log2((composition + eps).div(ct_global + eps, axis=1))
marker_score = composition * log2fc.clip(lower=0)

print(f"Cell types: {composition.shape[1]}, neighborhoods: {composition.shape[0]}")

# Niche order — cluster on composition
nhood_order = sorted(composition.index, key=lambda x: int(x))
if len(nhood_order) > 2:
    Z = linkage(np.sqrt(composition.loc[nhood_order].values),
                method="average", metric="euclidean")
    nhood_order = [nhood_order[i] for i in leaves_list(Z)]

# Cell type order — by best-niche assignment under marker score, then by score
nhood_index_map  = {nh: i for i, nh in enumerate(nhood_order)}
best_niche_per_ct = marker_score.loc[nhood_order].idxmax(axis=0)
best_score_per_ct = marker_score.loc[nhood_order].max(axis=0)
ct_table = pd.DataFrame({
    "best_niche_idx": best_niche_per_ct.map(nhood_index_map),
    "best_score":     best_score_per_ct,
})
ct_order = ct_table.sort_values(
    by=["best_niche_idx", "best_score"],
    ascending=[True, False],
).index.tolist()

# Reorder for plotting
color_M = marker_score.loc[nhood_order, ct_order].values
size_M  = specificity .loc[nhood_order, ct_order].values

# Dot sizing
size_scale = 220.0
spec_max   = specificity.values.max()
dot_areas  = size_M / max(spec_max, 1e-9) * size_scale
dot_areas  = np.maximum(dot_areas, 3)

# Layout
n_niches = composition.shape[0]
n_cts    = len(ct_order)
fig_w    = max(14.0, 0.22 * n_cts + 4.0)
fig_h    = max(3.5,  0.32 * n_niches + 2.5)
fig, ax  = plt.subplots(figsize=(fig_w, fig_h))

xx, yy = np.meshgrid(np.arange(n_cts), np.arange(n_niches))

# Faint grid through dot centers
for x in range(n_cts):
    ax.axvline(x, color="#ededed", linewidth=0.4, zorder=1)
for y in range(n_niches):
    ax.axhline(y, color="#ededed", linewidth=0.4, zorder=1)

sc = ax.scatter(
    xx.ravel(), yy.ravel(),
    s=dot_areas.ravel(),
    c=color_M.ravel(),
    cmap="RdPu", vmin=0, vmax=color_M.max(),
    edgecolors="#333333", linewidths=0.3,
    zorder=3,
)

ax.set_xticks(range(n_cts))
ax.set_xticklabels(ct_order, fontsize=9, rotation=45, ha="right",
                   rotation_mode="anchor")
ax.set_xlabel("Cell type", fontsize=11, labelpad=8)
ax.set_yticks(range(n_niches))
ax.set_yticklabels([f"N{nh_id}" for nh_id in nhood_order],
                   fontsize=10, fontweight="semibold")
ax.set_ylabel("Neighborhood", fontsize=10, labelpad=8)
ax.set_xlim(-0.5, n_cts - 0.5)
ax.set_ylim(-0.5, n_niches - 0.5)
ax.invert_yaxis()
ax.tick_params(length=0)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)

ax.set_title(
    "Cell-type markers per neighborhood (all cell types)",
    fontsize=12, fontweight="semibold", loc="left", pad=8,
)

cbar = plt.colorbar(sc, ax=ax, fraction=0.012, pad=0.01,
                    shrink=0.85, anchor=(0.0, 0.5))
cbar.set_label("Marker score\n(composition × log₂ FC⁺)", fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Specificity size legend
candidates = [0.05, 0.1, 0.2, 0.4, 0.7]
legend_vals = [v for v in candidates if v <= spec_max * 1.05]
if not legend_vals:
    legend_vals = [round(spec_max * f, 2) for f in [0.25, 0.5, 1.0]]
legend_handles = [
    plt.scatter([], [],
                s=v / max(spec_max, 1e-9) * size_scale,
                c="#7a7a7a", edgecolors="#333333", linewidths=0.3,
                label=f"{v:.2f}")
    for v in legend_vals
]
sleg = ax.legend(
    handles=legend_handles,
    loc="upper left", bbox_to_anchor=(1.06, 1.0),
    title="Specificity\n(fraction of cell type)",
    title_fontsize=8, fontsize=8,
    frameon=False, labelspacing=0.7, borderpad=0.3,
    handletextpad=0.6,
)
sleg.get_title().set_multialignment("left")
ax.add_artist(sleg)

plt.tight_layout()
plt.savefig("celltype_neighborhood_markers_all.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: celltype_neighborhood_markers_all.png")

adata.uns[f"{nhood_key}_marker_score"] = marker_score
print(f"Marker score table stored in adata.uns['{nhood_key}_marker_score']")

### Subset plot of cell composition

In [ ]:
# %% [markdown]
# ### Cell-type marker dotplot — top markers from selected neighborhoods
#
# Same marker score (composition × log2 FC⁺) as the all-cell-types version,
# but cell types are restricted to the union of top-N markers from a chosen
# subset of neighborhoods. Use this to filter out poorly-characterized niches
# (here: niches 0 and 6) from contributing cell types to the plot, while
# still showing all niche rows so the reader can see how the selected
# markers behave globally.

# %%
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list

# User config
nhood_key        = "celltype_neighborhood"
c2l_prefix       = "meanscell_abundance_w_sf_"
curation_col     = "celltype_level4_curated"
EXCLUDE_PATTERNS = ["doublet", "other", "unknown"]

TOP_N_PER_NICHE   = 5                                # markers per included niche
INCLUDED_NICHES   = ["1", "2", "3", "4", "5", "7", "8", "9"]   # niches whose top-N contribute
SHOW_EXCLUDED_ROWS = False                            # if True, plot all niche rows;
                                                     # if False, only included niches appear

# Sanity checks
if nhood_key not in adata.obs.columns:
    raise ValueError(f"{nhood_key} not found in adata.obs")
if "celltype_annotation_key" not in adata.uns:
    raise ValueError("adata.uns['celltype_annotation_key'] not found.")

celltype_key = adata.uns["celltype_annotation_key"]

# Pull abundance and translate level-4 → curated names
c2l_cols   = [c for c in adata.obs.columns if c.startswith(c2l_prefix)]
lvl4_names = pd.Index([c.replace(c2l_prefix, "") for c in c2l_cols])
counts     = adata.obs[c2l_cols].copy()
counts.columns = lvl4_names

lvl4_in_key  = lvl4_names.intersection(celltype_key.index)
counts       = counts[lvl4_in_key]
curated_map  = celltype_key.loc[lvl4_in_key, curation_col].astype(str).fillna("Unknown")

keep_mask = ~curated_map.str.lower().apply(
    lambda x: any(pat in x for pat in EXCLUDE_PATTERNS)
)
counts      = counts.loc[:, keep_mask.values]
curated_map = curated_map[keep_mask]

counts_curated = counts.T.groupby(curated_map).sum().T

# Spots with valid neighborhood and abundance
nh = adata.obs[nhood_key].astype("string")
valid_mask = nh.notna() & (counts_curated.sum(axis=1) > 0)
counts_curated = counts_curated.loc[valid_mask]
nh_valid = nh.loc[valid_mask]

# Core matrices
abund_per_nh = counts_curated.groupby(nh_valid, observed=True).sum()
composition  = abund_per_nh.div(abund_per_nh.sum(axis=1), axis=0).fillna(0)
specificity  = abund_per_nh.div(abund_per_nh.sum(axis=0), axis=1).fillna(0)
ct_global    = abund_per_nh.sum(axis=0) / abund_per_nh.values.sum()
eps          = 1e-6
log2fc       = np.log2((composition + eps).div(ct_global + eps, axis=1))
marker_score = composition * log2fc.clip(lower=0)

# Validate that included niches exist
missing = [n for n in INCLUDED_NICHES if n not in marker_score.index]
if missing:
    raise ValueError(f"INCLUDED_NICHES contains values not in data: {missing}")

# Pick top N markers per included niche, take the union
top_celltypes = set()
for nh_id in INCLUDED_NICHES:
    top_n = marker_score.loc[nh_id].sort_values(ascending=False).head(TOP_N_PER_NICHE)
    top_celltypes.update(top_n.index)
print(f"Cell types in plot: {len(top_celltypes)} "
      f"(top {TOP_N_PER_NICHE} from {len(INCLUDED_NICHES)} niches)")

# Niche row order — cluster on composition (subset to plot rows)
all_niches = sorted(composition.index, key=lambda x: int(x))
plot_niches = all_niches if SHOW_EXCLUDED_ROWS else INCLUDED_NICHES

# Use the full-niche clustering for row order so it's stable across runs
if len(all_niches) > 2:
    Z = linkage(np.sqrt(composition.loc[all_niches].values),
                method="average", metric="euclidean")
    full_order = [all_niches[i] for i in leaves_list(Z)]
else:
    full_order = all_niches
nhood_order = [n for n in full_order if n in plot_niches]

# Cell type order — by best-niche assignment, restricted to included niches
included_idx_map = {nh: i for i, nh in enumerate(INCLUDED_NICHES)}
ms_included = marker_score.loc[INCLUDED_NICHES, list(top_celltypes)]
best_niche_per_ct = ms_included.idxmax(axis=0)
best_score_per_ct = ms_included.max(axis=0)
ct_table = pd.DataFrame({
    "best_niche_idx": best_niche_per_ct.map(included_idx_map),
    "best_score":     best_score_per_ct,
})
ct_order = ct_table.sort_values(
    by=["best_niche_idx", "best_score"],
    ascending=[True, False],
).index.tolist()

# Reorder for plotting
color_M = marker_score.loc[nhood_order, ct_order].values
size_M  = specificity .loc[nhood_order, ct_order].values

# Dot sizing — anchor to specificity max over the cell types we're plotting
# Power transform with exponent > 1 expands the differential: small values
# get smaller, large values get larger. Combined with a higher size_scale
# and lower minimum floor, this gives a much more dramatic size range.
size_scale  = 400.0
size_power  = 1.6
size_floor  = 1.0
spec_max    = specificity.loc[nhood_order, ct_order].values.max()
dot_areas   = (size_M / max(spec_max, 1e-9)) ** size_power * size_scale
dot_areas   = np.maximum(dot_areas, size_floor)

# Layout
n_niches = len(nhood_order)
n_cts    = len(ct_order)
fig_w    = max(10.0, 0.30 * n_cts + 4.0)
fig_h    = max(3.0,  0.32 * n_niches + 2.5)
fig, ax  = plt.subplots(figsize=(fig_w, fig_h))

xx, yy = np.meshgrid(np.arange(n_cts), np.arange(n_niches))

# Faint grid through dot centers
for x in range(n_cts):
    ax.axvline(x, color="#ededed", linewidth=0.4, zorder=1)
for y in range(n_niches):
    ax.axhline(y, color="#ededed", linewidth=0.4, zorder=1)

# Dim rows that are excluded (visual cue that they didn't contribute markers)
if SHOW_EXCLUDED_ROWS:
    for i, nh_id in enumerate(nhood_order):
        if nh_id not in INCLUDED_NICHES:
            ax.axhspan(i - 0.5, i + 0.5, color="#fafafa", zorder=0)

sc = ax.scatter(
    xx.ravel(), yy.ravel(),
    s=dot_areas.ravel(),
    c=color_M.ravel(),
    cmap="RdPu", vmin=0, vmax=color_M.max(),
    edgecolors="#333333", linewidths=0.3,
    zorder=3,
)

ax.set_xticks(range(n_cts))
ax.set_xticklabels(ct_order, fontsize=9, rotation=45, ha="right",
                   rotation_mode="anchor")
ax.set_xlabel("Cell type", fontsize=11, labelpad=8)
ax.set_yticks(range(n_niches))
yticklabels = [
    f"N{nh_id}" + ("" if nh_id in INCLUDED_NICHES else " (excl.)")
    for nh_id in nhood_order
]
ax.set_yticklabels(yticklabels, fontsize=10, fontweight="semibold")
ax.set_ylabel("Neighborhood", fontsize=10, labelpad=8)
ax.set_xlim(-0.5, n_cts - 0.5)
ax.set_ylim(-0.5, n_niches - 0.5)
ax.invert_yaxis()
ax.tick_params(length=0)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)

title = (
    f"Cell-type markers per neighborhood "
    f"(top {TOP_N_PER_NICHE} from N{', N'.join(INCLUDED_NICHES)})"
)
ax.set_title(title, fontsize=12, fontweight="semibold", loc="left", pad=8)

cbar = plt.colorbar(sc, ax=ax, fraction=0.012, pad=0.01,
                    shrink=0.85, anchor=(0.0, 0.5))
cbar.set_label("Marker score\n(composition × log₂ FC⁺)", fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Specificity size legend
candidates = [0.05, 0.1, 0.2, 0.4, 0.7]
legend_vals = [v for v in candidates if v <= spec_max * 1.05]
if not legend_vals:
    legend_vals = [round(spec_max * f, 2) for f in [0.25, 0.5, 1.0]]
legend_handles = [
    plt.scatter([], [],
                s=(v / max(spec_max, 1e-9)) ** size_power * size_scale,
                c="#7a7a7a", edgecolors="#333333", linewidths=0.3,
                label=f"{v:.2f}")
    for v in legend_vals
]
sleg = ax.legend(
    handles=legend_handles,
    loc="upper left", bbox_to_anchor=(1.06, 1.0),
    title="Specificity\n(fraction of cell type)",
    title_fontsize=8, fontsize=8,
    frameon=False, labelspacing=0.7, borderpad=0.3,
    handletextpad=0.6,
)
sleg.get_title().set_multialignment("left")
ax.add_artist(sleg)

plt.tight_layout()
plt.savefig("celltype_neighborhood_markers_topN.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: celltype_neighborhood_markers_topN.png")

### Distribution of Neighborhoods Across Cohort

In [ ]:
# %% [markdown]
# ### Cell-type neighborhood distribution across patients

# %%
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.colors import to_hex

# User config
nhood_key            = "celltype_neighborhood"
sample_key           = "sample"
normalize            = True
min_spots_per_sample = 200
INCLUDED_NICHES      = None          # None = all niches; or a list like ["1","2","3","4","5","7","8","9"]
cluster_samples      = True
cluster_neighborhoods = True
cmap_name            = "tab20"
fig_width, fig_height = 18, 7
legend_ncol          = 2
save_dir             = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Notebooks"

# Sanity checks
for c in (sample_key, nhood_key):
    if c not in adata.obs.columns:
        raise KeyError(f"{c} not in adata.obs")

if not adata.obs[nhood_key].dtype.name == "category":
    adata.obs[nhood_key] = adata.obs[nhood_key].astype("category")

# Build sample × neighborhood count matrix
df    = adata.obs[[sample_key, nhood_key]].dropna()
pivot = df.groupby([sample_key, nhood_key], observed=True).size().unstack(fill_value=0)

# Drop low-coverage samples
n_before = len(pivot)
pivot = pivot.loc[pivot.sum(axis=1) >= min_spots_per_sample]
print(f"Samples: {len(pivot)} of {n_before} retained "
      f"(min {min_spots_per_sample} spots per sample)")
if pivot.empty:
    raise ValueError("No samples retained after filtering.")

# Optionally restrict to a subset of neighborhoods
if INCLUDED_NICHES is not None:
    pivot = pivot[[c for c in pivot.columns if str(c) in INCLUDED_NICHES]]
    print(f"Restricted to {len(INCLUDED_NICHES)} neighborhoods: {INCLUDED_NICHES}")

# Normalize within each sample so bars sum to 1 (proportion view)
if normalize:
    row_sums = pivot.sum(axis=1).replace(0, np.nan)
    pivot    = pivot.div(row_sums, axis=0).fillna(0)

# Cluster neighborhoods (cosine distance — appropriate for compositional data)
if cluster_neighborhoods and pivot.shape[1] > 2:
    Zc          = linkage(pivot.T.values.astype(np.float32),
                          method="average", metric="cosine")
    order_cols  = dendrogram(Zc, no_plot=True)["leaves"]
    pivot       = pivot.iloc[:, order_cols]
else:
    pivot = pivot.loc[:, pivot.sum(0).sort_values(ascending=False).index]

# Cluster samples (cosine distance on niche distribution)
if cluster_samples and pivot.shape[0] > 2:
    Zs          = linkage(pivot.values.astype(np.float32),
                          method="average", metric="cosine")
    order_rows  = dendrogram(Zs, no_plot=True)["leaves"]
    pivot       = pivot.iloc[order_rows]

# Use colors consistent with the spatial plots if already cached, else generate
cats_key   = f"{nhood_key}_categories"
colors_key = f"{nhood_key}_colors"

if cats_key in adata.uns and colors_key in adata.uns:
    cats_global   = list(adata.uns[cats_key])
    colors_global = list(adata.uns[colors_key])
    cm = dict(zip(cats_global, colors_global))
else:
    cats_global   = adata.obs[nhood_key].cat.categories.astype(str).tolist()
    cmap          = plt.get_cmap(cmap_name, len(cats_global))
    colors_global = [to_hex(cmap(i)) for i in range(len(cats_global))]
    adata.uns[cats_key]   = cats_global
    adata.uns[colors_key] = colors_global
    cm = dict(zip(cats_global, colors_global))

colors = [cm.get(str(col), "#999999") for col in pivot.columns]

# Plot
fig, ax = plt.subplots(figsize=(fig_width, fig_height))
x      = np.arange(pivot.shape[0])
bottom = np.zeros(pivot.shape[0], dtype=float)

for col, color in zip(pivot.columns, colors):
    vals = pivot[col].to_numpy()
    ax.bar(x, vals, bottom=bottom, width=1.0, color=color,
           edgecolor="none", label=f"N{col}")
    bottom += vals

ax.set_xlim(-0.5, len(x) - 0.5)
ax.set_ylim(0, 1.0 if normalize else None)
ax.set_ylabel("Proportion of spots" if normalize else "Spot count", fontsize=11)
ax.set_xlabel(f"Patients (n={len(x)}, hierarchically clustered)", fontsize=11)

# Sparse x-tick labels — for cohorts of 200+ samples, dense labels become unreadable
step = max(1, len(x) // 30)
ax.set_xticks(x[::step])
ax.set_xticklabels(pivot.index[::step], rotation=90, fontsize=7)
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(length=0)
ax.grid(False)
for spine_name in ["top", "right"]:
    ax.spines[spine_name].set_visible(False)

ax.set_title(
    "Cell-type neighborhood composition per patient",
    fontsize=12, fontweight="semibold", loc="left", pad=8,
)

ax.legend(
    loc="center left", bbox_to_anchor=(1.0, 0.5),
    frameon=False, ncol=legend_ncol, fontsize=9,
    title="Neighborhood", title_fontsize=10,
    handlelength=1.0, handleheight=1.0, labelspacing=0.4,
)

plt.tight_layout()
out_path = f"{save_dir}/celltype_neighborhood_per_patient.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

# Save the matrix for downstream use (e.g. correlation with clinical variables)
matrix_path = f"{save_dir}/celltype_neighborhoods_by_patient_stacked_matrix.csv"
pivot.to_csv(matrix_path)
print(f"\nSaved figure: {out_path}")
print(f"Saved matrix ({pivot.shape[0]} samples × {pivot.shape[1]} neighborhoods): {matrix_path}")